In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import TransformedTargetRegressor
import time

# 1. Read the train, valid and test dataframes that were already stratified and stored

In [4]:
train_df = pd.read_csv(r'C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\processed\stratified_sample_rng42\train_df.csv', index_col=0)
valid_df = pd.read_csv(r'C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\processed\stratified_sample_rng42\valid_df.csv', index_col=0)
test_df = pd.read_csv(r'C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\processed\stratified_sample_rng42\test_df.csv', index_col=0)
print('train_df.shape', train_df.shape)
print('valid_df.shape', valid_df.shape)
print('test_df.shape', test_df.shape)

train_df.shape (480000, 12)
valid_df.shape (150000, 12)
test_df.shape (150000, 12)


# 3. Create a function run Experiments efficiently

In [5]:
def run_xgb_experiment(
    name,
    train_df,
    valid_df,
    test_df,
    numeric_features,
    categorical_features,
    target_col="fare_amount",
    log_target=False,
    params=None,
):
    if params is None:
        params = dict(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        )

    X_train = train_df[numeric_features + categorical_features].copy()
    y_train = train_df[target_col].copy()

    X_valid = valid_df[numeric_features + categorical_features].copy()
    y_valid = valid_df[target_col].copy()

    X_test = test_df[numeric_features + categorical_features].copy()
    y_test = test_df[target_col].copy()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
        ("to_string", FunctionTransformer(lambda x: x.astype(str))),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    model = XGBRegressor(**params)

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    if log_target:
        y_train_fit = np.log1p(y_train)
        pipeline.fit(X_train, y_train_fit)

        valid_pred = np.expm1(pipeline.predict(X_valid))
        test_pred = np.expm1(pipeline.predict(X_test))

        valid_pred = np.clip(valid_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)
    else:
        pipeline.fit(X_train, y_train)
        valid_pred = pipeline.predict(X_valid)
        test_pred = pipeline.predict(X_test)

    valid_mae = mean_absolute_error(y_valid, valid_pred)
    valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    return {
        "experiment": name,
        "valid_mae": valid_mae,
        "valid_rmse": valid_rmse,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
    }

In [6]:
import numpy as np
import pandas as pd

train_base = train_df.copy()
valid_base = valid_df.copy()
test_base = test_df.copy()

# Set up the feeatures required for different experiments

In [7]:
def prepare_experiment_dfs(
    train_df,
    valid_df,
    test_df,
    use_pu_do=True,
    use_pu_do_pair=False,
    use_route_frequency=False,
    use_airport_flags=False,
    use_cyclical=False,
):
    train = train_df.copy()
    valid = valid_df.copy()
    test = test_df.copy()

    # ------------------------------
    # Base features
    # ------------------------------
    numeric_features = [
        "trip_distance",
        "passenger_count",
        "pickup_hour",
        "pickup_weekday",
        "pickup_month",
    ]

    categorical_features = [
        "VendorID",
        "RatecodeID",
        "store_and_fwd_flag",
    ]

    if use_pu_do:
        categorical_features += ["PULocationID", "DOLocationID"]

    # ------------------------------
    # PU_DO_pair
    # ------------------------------
    if use_pu_do_pair:
        for df in [train, valid, test]:
            df["PU_DO_pair"] = (
                df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
            )
        categorical_features.append("PU_DO_pair")

    # ------------------------------
    # Route frequency (train-derived only)
    # ------------------------------
    if use_route_frequency:
        for df in [train, valid, test]:
            df["route"] = (
                df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
            )

        route_count_map = train["route"].value_counts().to_dict()

        train["route_train_count"] = train["route"].map(route_count_map).fillna(0)
        valid["route_train_count"] = valid["route"].map(route_count_map).fillna(0)
        test["route_train_count"] = test["route"].map(route_count_map).fillna(0)

        train["route_train_count_log"] = np.log1p(train["route_train_count"])
        valid["route_train_count_log"] = np.log1p(valid["route_train_count"])
        test["route_train_count_log"] = np.log1p(test["route_train_count"])

        numeric_features.append("route_train_count_log")

    # ------------------------------
    # Airport flags
    # ------------------------------
    if use_airport_flags:
        AIRPORT_ZONES = {
            "JFK": 132,
            "LGA": 138,
            "EWR": 1,
        }

        for df in [train, valid, test]:
            df["is_jfk_trip"] = (
                (df["PULocationID"] == AIRPORT_ZONES["JFK"]) |
                (df["DOLocationID"] == AIRPORT_ZONES["JFK"])
            ).astype(int)

            df["is_lga_trip"] = (
                (df["PULocationID"] == AIRPORT_ZONES["LGA"]) |
                (df["DOLocationID"] == AIRPORT_ZONES["LGA"])
            ).astype(int)

            df["is_ewr_trip"] = (
                (df["PULocationID"] == AIRPORT_ZONES["EWR"]) |
                (df["DOLocationID"] == AIRPORT_ZONES["EWR"])
            ).astype(int)

            df["is_any_airport_trip"] = (
                df["is_jfk_trip"] | df["is_lga_trip"] | df["is_ewr_trip"]
            ).astype(int)

        numeric_features += [
            "is_jfk_trip",
            "is_lga_trip",
            "is_ewr_trip",
            "is_any_airport_trip",
        ]

    # ------------------------------
    # Cyclical encoding
    # ------------------------------
    if use_cyclical:
        for df in [train, valid, test]:
            df["pickup_hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
            df["pickup_hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)

            df["pickup_month_sin"] = np.sin(2 * np.pi * df["pickup_month"] / 12)
            df["pickup_month_cos"] = np.cos(2 * np.pi * df["pickup_month"] / 12)

        # replace raw hour/month with cyclical representation
        numeric_features = [
            f for f in numeric_features if f not in ["pickup_hour", "pickup_month"]
        ]
        numeric_features += [
            "pickup_hour_sin",
            "pickup_hour_cos",
            "pickup_month_sin",
            "pickup_month_cos",
        ]

    return train, valid, test, numeric_features, categorical_features

# Experiment Defining Function

In [8]:
def run_xgb_experiment(
    name,
    train_df,
    valid_df,
    test_df,
    numeric_features,
    categorical_features,
    target_col="fare_amount",
    log_target=False,
    params=None,
):
    if params is None:
        params = dict(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        )

    X_train = train_df[numeric_features + categorical_features].copy()
    y_train = train_df[target_col].copy()

    X_valid = valid_df[numeric_features + categorical_features].copy()
    y_valid = valid_df[target_col].copy()

    X_test = test_df[numeric_features + categorical_features].copy()
    y_test = test_df[target_col].copy()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
        ("to_string", FunctionTransformer(lambda x: x.astype(str))),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    model = XGBRegressor(**params)

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    if log_target:
        y_train_fit = np.log1p(y_train)
        pipeline.fit(X_train, y_train_fit)

        valid_pred = np.expm1(pipeline.predict(X_valid))
        test_pred = np.expm1(pipeline.predict(X_test))

        valid_pred = np.clip(valid_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)
    else:
        pipeline.fit(X_train, y_train)
        valid_pred = pipeline.predict(X_valid)
        test_pred = pipeline.predict(X_test)

    valid_mae = mean_absolute_error(y_valid, valid_pred)
    valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    return {
        "experiment": name,
        "valid_mae": valid_mae,
        "valid_rmse": valid_rmse,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
    }

# Run the experiments

In [9]:
experiment_specs = [
    {
        "name": "baseline_pu_do",
        "use_pu_do": True,
        "use_pu_do_pair": False,
        "use_route_frequency": False,
        "use_airport_flags": False,
        "use_cyclical": False,
        "log_target": False,
    },
    {
        "name": "with_pu_do_pair",
        "use_pu_do": True,
        "use_pu_do_pair": True,
        "use_route_frequency": False,
        "use_airport_flags": False,
        "use_cyclical": False,
        "log_target": False,
    },
    {
        "name": "with_route_frequency",
        "use_pu_do": True,
        "use_pu_do_pair": False,
        "use_route_frequency": True,
        "use_airport_flags": False,
        "use_cyclical": False,
        "log_target": False,
    },
    {
        "name": "without_pu_do",
        "use_pu_do": False,
        "use_pu_do_pair": False,
        "use_route_frequency": False,
        "use_airport_flags": False,
        "use_cyclical": False,
        "log_target": False,
    },
    {
        "name": "baseline_plus_cyclical",
        "use_pu_do": True,
        "use_pu_do_pair": False,
        "use_route_frequency": False,
        "use_airport_flags": False,
        "use_cyclical": True,
        "log_target": False,
    },
    {
        "name": "baseline_log_target",
        "use_pu_do": True,
        "use_pu_do_pair": False,
        "use_route_frequency": False,
        "use_airport_flags": False,
        "use_cyclical": False,
        "log_target": True,
    },
    {
        "name": "airport_flags_only_no_log",
        "use_pu_do": True,
        "use_pu_do_pair": False,
        "use_route_frequency": False,
        "use_airport_flags": True,
        "use_cyclical": False,
        "log_target": False,
    },
    {
        "name": "airport_flags_plus_log_target",
        "use_pu_do": True,
        "use_pu_do_pair": False,
        "use_route_frequency": False,
        "use_airport_flags": True,
        "use_cyclical": False,
        "log_target": True,
    },
    {
        "name": "airport_flags_plus_pu_do_pair_no_log",
        "use_pu_do": True,
        "use_pu_do_pair": True,
        "use_route_frequency": False,
        "use_airport_flags": True,
        "use_cyclical": False,
        "log_target": False,
    },
    {
        "name": "airport_flags_plus_pu_do_pair_plus_log_target",
        "use_pu_do": True,
        "use_pu_do_pair": True,
        "use_route_frequency": False,
        "use_airport_flags": True,
        "use_cyclical": False,
        "log_target": True,
    },
]

print('done')

done


In [10]:
results = []

for spec in experiment_specs:
    print(f"Running: {spec['name']}")

    tr, va, te, num_feats, cat_feats = prepare_experiment_dfs(
        train_base,
        valid_base,
        test_base,
        use_pu_do=spec["use_pu_do"],
        use_pu_do_pair=spec["use_pu_do_pair"],
        use_route_frequency=spec["use_route_frequency"],
        use_airport_flags=spec["use_airport_flags"],
        use_cyclical=spec["use_cyclical"],
    )

    res = run_xgb_experiment(
        name=spec["name"],
        train_df=tr,
        valid_df=va,
        test_df=te,
        numeric_features=num_feats,
        categorical_features=cat_feats,
        log_target=spec["log_target"],
    )

    results.append(res)

results_df = pd.DataFrame(results).sort_values("test_mae")
#results_df

Running: baseline_pu_do
Running: with_pu_do_pair
Running: with_route_frequency
Running: without_pu_do
Running: baseline_plus_cyclical
Running: baseline_log_target
Running: airport_flags_only_no_log
Running: airport_flags_plus_log_target
Running: airport_flags_plus_pu_do_pair_no_log
Running: airport_flags_plus_pu_do_pair_plus_log_target


In [11]:
results_df.style.format({
    "valid_mae": "{:.4f}",
    "valid_rmse": "{:.4f}",
    "test_mae": "{:.4f}",
    "test_rmse": "{:.4f}",
})

,experiment,valid_mae,valid_rmse,test_mae,test_rmse
5,baseline_log_target,2.2423,4.9067,2.7771,5.6806
7,airport_flags_plus_log_target,2.2467,4.9316,2.7816,5.6878
6,airport_flags_only_no_log,2.2199,4.7503,2.8352,5.7424
4,baseline_plus_cyclical,2.2250,4.7717,2.8361,5.7622
0,baseline_pu_do,2.2209,4.7544,2.8398,5.7670
9,airport_flags_plus_pu_do_pair_plus_log_target,2.2735,4.9646,2.8417,5.8126
1,with_pu_do_pair,2.2448,4.7909,2.8759,5.7854
8,airport_flags_plus_pu_do_pair_no_log,2.2420,4.8112,2.8768,5.8087
2,with_route_frequency,2.2722,4.9042,2.9983,6.1952
3,without_pu_do,2.4095,5.2251,3.0685,6.2832


In [12]:
# Error analysis

# baseline_log_target error analysis

In [13]:
spec = {
    "name": "baseline_log_target",
    "use_pu_do": True,
    "use_pu_do_pair": False,
    "use_route_frequency": False,
    "use_airport_flags": False,
    "use_cyclical": False,
    "log_target": True
}

target_col="fare_amount"
train_df, valid_df, test_df, numeric_features, categorical_features = prepare_experiment_dfs(
        train_base,
        valid_base,
        test_base,
        use_pu_do=spec["use_pu_do"],
        use_pu_do_pair=spec["use_pu_do_pair"],
        use_route_frequency=spec["use_route_frequency"],
        use_airport_flags=spec["use_airport_flags"],
        use_cyclical=spec["use_cyclical"],
    )

In [14]:
params = dict(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

# IMPORTANT: use tr / va / te, not train_df / valid_df / test_df
X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df[target_col].copy()

X_valid = valid_df[numeric_features + categorical_features].copy()
y_valid = valid_df[target_col].copy()

X_test = test_df[numeric_features + categorical_features].copy()
y_test = test_df[target_col].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("to_string", FunctionTransformer(lambda x: x.astype(str))),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = XGBRegressor(**params)

wrapped_model = TransformedTargetRegressor(
    regressor=model,
    func=np.log1p,
    inverse_func=np.expm1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", wrapped_model),
])

pipeline.fit(X_train, y_train)
print("Model fit complete.")

Model fit complete.


In [19]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")


def run_full_diagnostics(
    pipeline,
    X_eval,
    y_eval,
    df_full_context,
    month_col="pickup_month",
    date_col="tpep_pickup_datetime",
    distance_col="trip_distance",
    save_dir=r"C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots",
    img_size=(6, 4),
    dpi=150
):
    """
    Generate and save the following diagnostic plots:

    1. actual_vs_predicted.png
    2. residual_distribution.png
    3. error_vs_distance.png
    4. mae_by_month.png
    5. mean_residual_by_month.png
    6. mae_by_distance_bucket.png

    Returns
    -------
    results : pd.DataFrame
        Evaluation dataframe with predictions, residuals, errors, and context columns.
    metrics : dict
        Summary metrics like MAE, RMSE, bias.
    plot_paths : dict
        Mapping from logical plot name to saved file path.
    """

    # -----------------------------
    # Basic checks
    # -----------------------------
    if not isinstance(X_eval, pd.DataFrame):
        raise ValueError("X_eval must be a pandas DataFrame so we can preserve index and columns.")

    if distance_col not in X_eval.columns and distance_col not in df_full_context.columns:
        raise ValueError(f"'{distance_col}' not found in X_eval or df_full_context.")

    os.makedirs(save_dir, exist_ok=True)

    # -----------------------------
    # Predictions + core results df
    # -----------------------------
    y_pred = pipeline.predict(X_eval)

    results = X_eval.copy()
    results["actual"] = np.asarray(y_eval)
    results["predicted"] = np.asarray(y_pred)
    results["residual"] = results["predicted"] - results["actual"]   # signed error
    results["abs_error"] = np.abs(results["residual"])
    results["squared_error"] = results["residual"] ** 2

    # Bring back context columns from original dataframe if missing
    for col in [month_col, date_col, distance_col]:
        if col in df_full_context.columns and col not in results.columns:
            results[col] = df_full_context.loc[X_eval.index, col]

    # -----------------------------
    # Create a robust month label
    # -----------------------------
    if date_col in results.columns:
        results[date_col] = pd.to_datetime(results[date_col], errors="coerce")
        results["month_period"] = results[date_col].dt.to_period("M")
        results["month_label"] = results["month_period"].astype(str)
    elif month_col in results.columns:
        # fallback if datetime not present
        results["month_label"] = results[month_col].astype(str)
        results["month_period"] = results["month_label"]
    else:
        results["month_label"] = "unknown"
        results["month_period"] = "unknown"

    # -----------------------------
    # Helper functions
    # -----------------------------
    plot_paths = {}

    def save_current_fig(filename):
        path = os.path.join(save_dir, f"{filename}.png")
        plt.tight_layout()
        plt.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close()
        plot_paths[filename] = path

    def annotate_bars(ax, fmt="{:.2f}", rotation=0, fontsize=9):
        for p in ax.patches:
            height = p.get_height()
            if pd.notna(height):
                ax.annotate(
                    fmt.format(height),
                    (p.get_x() + p.get_width() / 2.0, height),
                    ha="center",
                    va="bottom",
                    fontsize=fontsize,
                    rotation=rotation,
                    xytext=(0, 4),
                    textcoords="offset points"
                )

    # -----------------------------
    # Plot 1: Actual vs Predicted
    # -----------------------------
    plt.figure(figsize=img_size)
    actual_q99 = results["actual"].quantile(0.99)
    pred_q99 = results["predicted"].quantile(0.99)
    identity_max = max(actual_q99, pred_q99, 50)

    plt.scatter(
        results["actual"],
        results["predicted"],
        alpha=0.12,
        s=6
    )
    plt.plot(
        [0, identity_max],
        [0, identity_max],
        linestyle="--",
        linewidth=2
    )
    plt.title("Actual vs Predicted Fare")
    plt.xlabel("Actual Fare ($)")
    plt.ylabel("Predicted Fare ($)")
    plt.xlim(0, identity_max)
    plt.ylim(0, identity_max)
    save_current_fig("actual_vs_predicted")

    # -----------------------------
    # Plot 2: Residual Distribution
    # -----------------------------
    plt.figure(figsize=img_size)
    lo = results["residual"].quantile(0.005)
    hi = results["residual"].quantile(0.995)

    sns.histplot(
        results["residual"].clip(lower=lo, upper=hi),
        bins=50,
        kde=True
    )
    plt.axvline(0, linestyle="--", linewidth=1.5)
    plt.title("Residual Distribution")
    plt.xlabel("Residual ($)")
    plt.ylabel("Count")
    save_current_fig("residual_distribution")

    # -----------------------------
    # Plot 3: Absolute Error vs Distance
    # -----------------------------
    plt.figure(figsize=img_size)
    dist_q99 = results[distance_col].quantile(0.99)
    plot_df = results.loc[results[distance_col] <= dist_q99].copy()

    plt.scatter(
        plot_df[distance_col],
        plot_df["abs_error"],
        alpha=0.12,
        s=6
    )
    plt.title("Absolute Error vs Trip Distance")
    plt.xlabel("Trip Distance (miles)")
    plt.ylabel("Absolute Error ($)")
    save_current_fig("error_vs_distance")

    # -----------------------------
    # Plot 4: MAE by Month
    # -----------------------------
    month_mae = (
        results.groupby(["month_period", "month_label"], observed=True)["abs_error"]
        .mean()
        .reset_index()
        .sort_values("month_period")
    )

    plt.figure(figsize=img_size)
    ax = sns.barplot(
        data=month_mae,
        x="month_label",
        y="abs_error"
    )
    plt.title("MAE by Month")
    plt.xlabel("Month")
    plt.ylabel("Mean Absolute Error ($)")
    plt.xticks(rotation=30)
    annotate_bars(ax)
    save_current_fig("mae_by_month")

    # -----------------------------
    # Plot 5: Mean Residual by Month
    # -----------------------------
    month_resid = (
        results.groupby(["month_period", "month_label"], observed=True)["residual"]
        .mean()
        .reset_index()
        .sort_values("month_period")
    )

    plt.figure(figsize=img_size)
    ax = sns.barplot(
        data=month_resid,
        x="month_label",
        y="residual"
    )
    plt.axhline(0, linestyle="--", linewidth=1.5)
    plt.title("Mean Residual by Month")
    plt.xlabel("Month")
    plt.ylabel("Mean Residual ($)")
    plt.xticks(rotation=30)
    annotate_bars(ax)
    save_current_fig("mean_residual_by_month")

    # -----------------------------
    # Plot 6: MAE by Distance Bucket
    # -----------------------------
    dist_bins = [0, 1, 3, 5, 10, 20, np.inf]
    dist_labels = ["0-1", "1-3", "3-5", "5-10", "10-20", "20+"]

    results["distance_bucket"] = pd.cut(
        results[distance_col],
        bins=dist_bins,
        labels=dist_labels,
        right=False
    )

    dist_mae = (
        results.groupby("distance_bucket", observed=True)["abs_error"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=img_size)
    ax = sns.barplot(
        data=dist_mae,
        x="distance_bucket",
        y="abs_error"
    )
    plt.title("MAE by Distance Bucket")
    plt.xlabel("Distance Bucket (miles)")
    plt.ylabel("Mean Absolute Error ($)")
    plt.xticks(rotation=30)
    annotate_bars(ax)
    save_current_fig("mae_by_distance_bucket")

    # -----------------------------
    # Summary metrics
    # -----------------------------
    mae = mean_absolute_error(results["actual"], results["predicted"])
    rmse = np.sqrt(mean_squared_error(results["actual"], results["predicted"]))
    bias = results["residual"].mean()

    metrics = {
        "mae": float(mae),
        "rmse": float(rmse),
        "mean_residual_bias": float(bias),
        "n_rows": int(len(results))
    }

    print("\n--- DIAGNOSTICS COMPLETE ---")
    print(f"Images saved to: {save_dir}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"Bias : {bias:.4f}")
    print("\nSaved plots:")
    for k, v in plot_paths.items():
        print(f"- {k}: {v}")

    return results, metrics, plot_paths

In [20]:
test_results, test_metrics, test_plot_paths = run_full_diagnostics(
    pipeline=pipeline,
    X_eval=X_test,
    y_eval=y_test,
    df_full_context=te,
    month_col="pickup_month",
    date_col="tpep_pickup_datetime",
    distance_col="trip_distance",
    save_dir=r"C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots"
)


--- DIAGNOSTICS COMPLETE ---
Images saved to: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots
MAE  : 2.7771
RMSE : 5.6806
Bias : -0.1355

Saved plots:
- actual_vs_predicted: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots\actual_vs_predicted.png
- residual_distribution: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots\residual_distribution.png
- error_vs_distance: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots\error_vs_distance.png
- mae_by_month: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots\mae_by_month.png
- mean_residual_by_month: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots\mean_residual_by_month.png
- mae_by_distance_bucket: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots\mae_by_distance_bucket.png


C:\Users\Rahul\AppData\Local\Temp\ipykernel_24600\1942417100.py:92: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=dist_mae, x="distance_bucket", y="abs_error", palette="Blues_d")



--- DIAGNOSTICS COMPLETE ---
Images saved to: C:\Users\Rahul\OneDrive\Desktop\Learning\Projects\nyc_taxi\plots
Final Metrics -> MAE: 2.78 | RMSE: 5.68


ValueError: too many values to unpack (expected 2)